# Test Case 4 — Reactor Trip: PLC Gates and Timeline Gate

## Purpose

Demonstrates two key hard-gate features using a realistic automatic reactor trip scenario:

1. **Timeline consistency gate** — eliminates `FM-CRD-MECHANICAL` automatically because the CRD
   position anomaly is timestamped **10 seconds after** the NI-4 trip initiator in the SOE log.
   The Allen `follows` temporal relation is identified from fixture timestamps; no analyst input required.

2. **Barrier logic / PLC gate** — confirms that the RPS barrier state is `held` (all CRDs inserted,
   core subcritical) and that the protection logic context was consulted during scoring of NI candidates.

## Scenario Summary

| Field | Value |
|---|---|
| **Event ID** | `E2026-03-10-001` |
| **Asset** | U1 Reactor Protection System |
| **Trip time** | 2026-03-10T14:22:03Z |
| **Trip initiator** | NI channel 4 (`U1-NI-RPS-4A`) — 112.3% rated, 105% setpoint |
| **Primary hypothesis** | `FM-NI-SPURIOUS` — spurious instrument signal (EMI + calibration offset) |
| **Competing hypotheses** | `FM-FW-TRANSIENT` (real flux excursion), `FM-CRD-MECHANICAL` (ruled out by timeline) |

## Show-stopper

Hypothesis C (`FM-CRD-MECHANICAL`) is a well-known cognitive trap in nuclear event investigation:
an analyst reviewing raw data under time pressure might note the CRD anomaly and incorrectly
assume it preceded the trip. The pipeline sidesteps this trap by reading the SOE timestamps
directly — the CRD position step at T+10s is placed **after** the NI trip signal, so the Allen
`follows` relation is identified and the timeline consistency gate eliminates this hypothesis
without human intervention.

In [ ]:
from __future__ import annotations
import json
import os
import sys
from pathlib import Path
from typing import Any, Dict, Optional

NOTEBOOK_ROOT = Path.cwd().resolve()
FIXTURE_DIR   = NOTEBOOK_ROOT / "fixtures"
OUTPUT_DIR    = NOTEBOOK_ROOT / "rca_runs_case_004"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Python path — adjust if the notebook is in a different location
rca_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if rca_root not in sys.path:
    sys.path.insert(0, rca_root)
dackar_root = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
if dackar_root not in sys.path:
    sys.path.insert(0, dackar_root)
shared_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "shared"))
if shared_dir not in sys.path:
    sys.path.insert(0, shared_dir)

from run_helpers import build_fixture_orchestrator, load_fixtures, run_rca, summarise_result
from assertion_helpers import (
    assert_primary_cause,
    assert_candidate_present,
    assert_hard_gate_failed,
    assert_data_coverage_status,
    assert_barrier_analysis_present,
    run_assertion_table,
)

print("Fixture dir :", FIXTURE_DIR)
print("Output dir  :", OUTPUT_DIR)

In [ ]:
## Load all fixtures
fixtures = load_fixtures(FIXTURE_DIR)

loaded_summary = {
    k: ("loaded" if v is not None else "absent")
    for k, v in fixtures.items()
    if k not in ("event", "telemetry_summary", "kg_context")
}
print("Required: event, telemetry_summary, kg_context — loaded")
print("Optional:", json.dumps(loaded_summary, indent=2))

# Sanity checks
assert fixtures["event"]["event_id"] == fixtures["telemetry_summary"]["event_id"], "event_id mismatch"
assert fixtures["event"]["asset_id"] == fixtures["telemetry_summary"]["asset_id"], "asset_id mismatch"
print("Fixture sanity checks passed.")

In [ ]:
## Build orchestrator
orchestrator = build_fixture_orchestrator(OUTPUT_DIR, top_k_candidates=5, enable_ishikawa=True)
print("Orchestrator built — causality engine v32, FileArtifactStore:", OUTPUT_DIR)

In [ ]:
## Run the RCA pipeline
result = run_rca(orchestrator, fixtures)
print("Pipeline run complete.")
summarise_result(result)

## Hard Gate Analysis

### Timeline Gate — FM-CRD-MECHANICAL ruled out

The SOE log places `U1-ZI-CRD-2214` position change at T+10s after the NI-4 trip initiator.
The Allen temporal analysis identifies this as `NI-4 trip PRECEDES CRD position change`,
meaning the CRD anomaly *follows* the trip — it is a consequence, not a cause.

### Barrier Logic Gate — PLC consulted for NI candidates

The protection_logic_context.json contains the RPS trip logic set. The pipeline consults this
when evaluating candidates that involve protection signals. The barrier state for the reactor
trip function is `held` — the SCRAM performed as designed.

In [ ]:
## Inspect hard gate results for each candidate
candidates = (result.get("causality_candidates") or {}).get("candidates") or []

print(f"Retained candidates: {len(candidates)}")
for cand in candidates:
    fm_id   = cand.get("failure_mode_id", "?")
    gates   = cand.get("hard_gates") or {}
    tl      = gates.get("timeline_consistency") or {}
    bl      = gates.get("barrier_logic") or {}
    scores  = cand.get("scores") or {}
    print(f"\n  [{fm_id}]")
    print(f"    composite            : {scores.get('composite', '?')}")
    print(f"    timeline_consistency : passed={tl.get('passed', '?')}")
    print(f"    barrier_logic        : plc_consulted={bl.get('plc_consulted', '?')}")
    print(f"    chain_position       : {cand.get('chain_position', '?')}")

In [ ]:
## Show CRD candidate rule-out detail (if present in decision trail)
decision_trail = result.get("decision_trail") or []
crd_entries = [e for e in decision_trail if "CRD" in str(e.get("candidate_id", ""))]
if crd_entries:
    print("CRD candidate decision trail entries:")
    for e in crd_entries:
        print(json.dumps(e, indent=2, default=str)[:800])
else:
    print("No CRD-specific entries in decision trail — candidate may have been filtered before trail.")

# Data coverage summary (navigate through source_families)
cov_summary = (result.get("run_manifest") or {}).get("artifacts", {}).get("data_coverage_summary") or {}
source_families = cov_summary.get("source_families") or {}
print("\nData coverage (source_families):")
for src, info in source_families.items():
    status = info.get("status") if isinstance(info, dict) else info
    print(f"  {src:40s}: {status}")

In [ ]:
## Run assertion table
assertions = [
    {
        "id": "A4-1",
        "desc": "FM-NI-SPURIOUS candidate present",
        "fn": lambda r: assert_candidate_present(r, "FM-NI-SPURIOUS"),
    },
    {
        "id": "A4-2",
        "desc": "Primary hypothesis is FM-NI-SPURIOUS",
        "fn": lambda r: assert_primary_cause(r, "FM-NI-SPURIOUS"),
    },
    {
        "id": "A4-3",
        "desc": "Environmental monitoring complete in coverage",
        "fn": lambda r: assert_data_coverage_status(r, "environmental_monitoring", "complete"),
    },
    {
        "id": "A4-4",
        "desc": "Protection logic context complete in coverage",
        "fn": lambda r: assert_data_coverage_status(r, "protection_logic_context", "complete"),
    },
    {
        "id": "A4-5",
        "desc": "SOE log complete in coverage",
        "fn": lambda r: assert_data_coverage_status(r, "soe_log", "complete"),
    },
]

run_assertion_table(result, assertions, label="TC-4 Assertions")

In [ ]:
## Save full result to disk
out_path = OUTPUT_DIR / "tc4_full_result.json"
with open(out_path, "w", encoding="utf-8") as fh:
    json.dump(result, fh, indent=2, default=str)
print(f"Full result saved to: {out_path}")